# Phần 6 – Tổng hợp nhận định và bàn giao
Nhóm 5 PTDL · Trực quan hóa & Phân tích thống kê

| Bước | Việc trong kế hoạch | Kết quả |
|---|---|---|
| 1 | Chạy lại toàn bộ notebook từ đầu | 01 → 05 không lỗi |
| 2 | Xuất 4 file CSV theo định dạng thống nhất | gộp `thong_ke_mo_ta.csv`, kiểm tra cả 4 file |
| 3 | Rà soát biểu đồ (màu, tiêu đề, đơn vị) | danh mục hình + bảng ảnh thu nhỏ |
| 4 | Chọn bộ hình cho báo cáo | `outputs/figures/bao_cao/Hinh_XX_*.png` |
| 5 | Số liệu dẫn chứng cho nhận định câu 1, 2, 3 | đọc trực tiếp từ 4 file CSV |

In [1]:
import sys, shutil, subprocess
from pathlib import Path
GOC = Path.cwd().parent
sys.path.insert(0, str(GOC))

import pandas as pd
from src import config as cf
from src.ket_qua import CAU_TRUC, xuat_ket_qua

KQ = cf.THU_MUC_KET_QUA
pd.set_option("display.max_colwidth", 90)

---
## 1. Chạy lại toàn bộ notebook 01 → 05
Mất khoảng 2 phút. Đặt `CHAY_LAI = False` nếu vừa chạy xong từng notebook và không muốn chạy lại.
Có thể chạy cùng việc này từ Terminal: `python chay_lai_toan_bo.py`.

In [2]:
CHAY_LAI = True
if CHAY_LAI:
    kq = subprocess.run([sys.executable, str(GOC / "chay_lai_toan_bo.py")], capture_output=True, text=True, cwd=GOC)
    print(kq.stdout or kq.stderr)

c:\Users\hoang\AppData\Local\Python\pythoncore-3.14-64\python.exe: can't open file 'c:\\Users\\hoang\\OneDrive\\NhÃ³m5-PTDL\\chay_lai_toan_bo.py': [Errno 2] No such file or directory



---
## 2. Bốn file bảng kết quả
### 2.1 Gộp thống kê mô tả (Phần 3: ROA, ROE + Phần 4: NIM, NPL, CAR)

In [3]:
p3 = pd.read_csv(KQ / "tam" / "thong_ke_mo_ta_p3.csv", encoding="utf-8-sig")
p4 = pd.read_csv(KQ / "tam" / "thong_ke_mo_ta_p4.csv", encoding="utf-8-sig")
mo_ta = pd.concat([p3, p4], ignore_index=True)

khoa = ["chi_so", "cap_do", "ma_nh", "nam", "loai_hinh_so_huu", "mau_du_lieu"]
assert not mo_ta.duplicated(khoa).any(), "Có dòng trùng khi gộp"
xuat_ket_qua(mo_ta, "thong_ke_mo_ta")
print(f"Đã lưu outputs/thong_ke_mo_ta.csv: {len(mo_ta)} dòng")
mo_ta.groupby(["chi_so", "cap_do"]).size().unstack(fill_value=0)

Đã lưu outputs/thong_ke_mo_ta.csv: 155 dòng


cap_do,loai_hinh_so_huu,ma_nh,ma_nh_nam,nam,toan_bo
chi_so,,,,,
car,2,8,0,4,1
nim,2,8,0,5,1
npl,2,8,0,5,1
roa,0,8,40,5,1
roe,0,8,40,5,1


### 2.2 Kiểm tra cả 4 file trước khi giao cho Hân
Đọc lại đúng như dashboard sẽ đọc, kiểm tra: đủ cột, đúng thứ tự, không trống ở cột bắt buộc.

In [4]:
BAT_BUOC = {
    "thong_ke_mo_ta": ["chi_so", "cap_do", "so_quan_sat", "trung_binh"],
    "xep_hang_nam": ["chi_so", "nam", "ma_nh", "gia_tri", "hang"],
    "kiem_dinh_nhom": ["chi_so", "phan_tich", "kiem_dinh", "p_value", "ket_luan"],
    "tuong_quan": ["chi_so", "phan_tich", "so_quan_sat", "pearson_r", "spearman_rho"],
}
dong = []
for ten, cot in CAU_TRUC.items():
    f = KQ / f"{ten}.csv"
    df = pd.read_csv(f, encoding="utf-8-sig")
    dong.append({"file": f.name, "so_dong": len(df), "so_cot": df.shape[1],
                 "dung_cot_dung_thu_tu": list(df.columns) == cot,
                 "cot_bat_buoc_khong_trong": df[BAT_BUOC[ten]].notna().all().all(),
                 "kich_thuoc_KB": round(f.stat().st_size / 1024, 1)})
kiem_tra = pd.DataFrame(dong)
assert kiem_tra["dung_cot_dung_thu_tu"].all() and kiem_tra["cot_bat_buoc_khong_trong"].all()
kiem_tra

,file,so_dong,so_cot,dung_cot_dung_thu_tu,cot_bat_buoc_khong_trong,kich_thuoc_KB
0,thong_ke_mo_ta.csv,155,13,True,True,10.8
1,xep_hang_nam.csv,80,8,True,True,4.0
2,kiem_dinh_nhom.csv,8,24,True,True,3.3
3,tuong_quan.csv,31,13,True,True,3.4


---
## 3. Rà soát biểu đồ
### 3.1 Danh mục hình

In [5]:
hinh = sorted(cf.THU_MUC_HINH.glob("p[2-5]_*.png"))
danh_muc = pd.DataFrame({"file": [h.name for h in hinh],
                         "phan": [h.name[:2].upper() for h in hinh]})
print(f"Tổng {len(hinh)} hình:", danh_muc["phan"].value_counts().sort_index().to_dict())

Tổng 0 hình: {}


### 3.2 Bảng ảnh thu nhỏ để soát bằng mắt
Lưu tại `outputs/ra_soat_hinh_<phần>.png`. Mở từng file và kiểm tra theo danh sách:
- Màu: mỗi ngân hàng một màu cố định ở mọi hình; quốc doanh xanh dương, tư nhân cam.
- Tiêu đề: tiếng Việt, có tên chỉ số + phạm vi (ngân hàng, năm).
- Đơn vị: trục có (%) hoặc (nghìn đồng); CAR ghi rõ "số công bố".
- Chú thích: tên ngân hàng/nhóm đầy đủ, không chồng chữ.

In [6]:
from PIL import Image, ImageDraw

def bang_anh(ds, ra, so_cot=4, rong=480):
    anh = []
    for f in ds:
        im = Image.open(f).convert("RGB")
        anh.append((f.name, im.resize((rong, int(im.height * rong / im.width)))))
    cao = max(i.height for _, i in anh) + 18
    so_hang = (len(anh) + so_cot - 1) // so_cot
    tam = Image.new("RGB", (so_cot * rong, so_hang * cao), "white")
    ve = ImageDraw.Draw(tam)
    for k, (ten, im) in enumerate(anh):
        x, y = (k % so_cot) * rong, (k // so_cot) * cao
        tam.paste(im, (x, y + 18))
        ve.text((x + 4, y + 3), ten, fill="red")
    tam.save(ra)
    return ra

for ph in ["p2", "p3", "p4", "p5"]:
    ds = [h for h in hinh if h.name.startswith(ph)]
    print(bang_anh(ds, KQ / f"ra_soat_hinh_{ph}.png"))

ValueError: max() iterable argument is empty

---
## 4. Bộ hình cho báo cáo
Chọn 13 hình chính, đánh số theo thứ tự xuất hiện trong mục báo cáo, chép vào `outputs/figures/bao_cao/`. Các hình còn lại vẫn trong `outputs/figures/` để dùng cho phụ lục hoặc slide.

In [ ]:
HINH_BAO_CAO = [
    ("p2_tong_quan_6_chi_so",     f"Tổng quan 6 chỉ số tài chính theo ngân hàng, Q1/{cf.NAM_BAT_DAU} – Q{cf.QUY_CUOI}/{cf.NAM_KET_THUC}"),
    ("p2_gia_chuan_hoa",          "Giá cổ phiếu điều chỉnh, chuẩn hóa = 100 tại phiên đầu năm 2021"),
    ("p2_su_kien_theo_nh",        "Số sự kiện phi tài chính theo ngân hàng và loại sự kiện"),
    ("p3_xu_huong_roe_nam",       "ROE năm của 8 ngân hàng, 2021–2025"),
    ("p3_bump_roe",               "Thứ hạng ROE năm qua các năm"),
    ("p3_bump_roa",               "Thứ hạng ROA năm qua các năm"),
    ("p4_hop_nhom_nim",           "NIM: quốc doanh và tư nhân"),
    ("p4_hop_nhom_npl",           "NPL: quốc doanh và tư nhân"),
    ("p4_duong_nhom_npl",         "NPL trung bình theo nhóm, có và không có VPB"),
    ("p4_hop_nhom_car",           "CAR công bố: quốc doanh và tư nhân"),
    ("p5_cot_tuong_quan_chinh",   "Tương quan của 6 chỉ số với lợi suất cổ phiếu quý"),
    ("p5_ma_tran_spearman",       "Ma trận tương quan Spearman"),
    ("p5_tuong_quan_so_sanh",     "Tương quan theo các cách kiểm tra lại"),
]
thu_muc_bc = cf.THU_MUC_HINH / "bao_cao"
if thu_muc_bc.exists():
    shutil.rmtree(thu_muc_bc)
thu_muc_bc.mkdir(parents=True)
dm = []
for i, (ten, chu_thich) in enumerate(HINH_BAO_CAO, 1):
    moi = f"Hinh_{i:02d}_{ten}.png"
    shutil.copy(cf.THU_MUC_HINH / f"{ten}.png", thu_muc_bc / moi)
    dm.append({"so_hinh": f"Hình {i}", "file": moi, "chu_thich": chu_thich})
dm = pd.DataFrame(dm)
dm.to_csv(thu_muc_bc / "danh_muc_hinh.csv", index=False, encoding="utf-8-sig")
dm

,so_hinh,file,chu_thich
0,Hình 1,Hinh_01_p2_tong_quan_6_chi_so.png,"Tổng quan 6 chỉ số tài chính theo ngân hàng, 2021–2025"
1,Hình 2,Hinh_02_p2_gia_chuan_hoa.png,"Giá cổ phiếu điều chỉnh, chuẩn hóa = 100 tại phiên đầu năm 2021"
2,Hình 3,Hinh_03_p2_su_kien_theo_nh.png,Số sự kiện phi tài chính theo ngân hàng và loại sự kiện
3,Hình 4,Hinh_04_p3_xu_huong_roe_nam.png,"ROE năm của 8 ngân hàng, 2021–2025"
4,Hình 5,Hinh_05_p3_bump_roe.png,Thứ hạng ROE năm qua các năm
5,Hình 6,Hinh_06_p3_bump_roa.png,Thứ hạng ROA năm qua các năm
6,Hình 7,Hinh_07_p4_hop_nhom_nim.png,NIM: quốc doanh và tư nhân
7,Hình 8,Hinh_08_p4_hop_nhom_npl.png,NPL: quốc doanh và tư nhân
8,Hình 9,Hinh_09_p4_duong_nhom_npl.png,"NPL trung bình theo nhóm, có và không có VPB"
9,Hình 10,Hinh_10_p4_hop_nhom_car.png,CAR công bố: quốc doanh và tư nhân


---
## 5. Số liệu dẫn chứng cho nhận định
Mọi con số trong mục báo cáo được đọc từ 4 file CSV – nếu dữ liệu thay đổi (ví dụ Hân sửa CAR), chạy lại notebook là có số mới.
### 5.1 Câu 1 – Sinh lời và độ ổn định

In [ ]:
xh = pd.read_csv(KQ / "xep_hang_nam.csv", encoding="utf-8-sig")
for cs in ["roe_nam", "roa_nam"]:
    b = xh[xh["chi_so"] == cs].pivot(index="ma_nh", columns="nam", values="hang")
    b["so_nam_hang_1"] = (b == 1).sum(axis=1)
    b["hang_tb"] = b.drop(columns="so_nam_hang_1").mean(axis=1)
    print(f"■ Thứ hạng {cs}")
    display(b.sort_values("hang_tb"))

■ Thứ hạng roe_nam


nam,2021,2022,2023,2024,2025,so_nam_hang_1,hang_tb
ma_nh,,,,,,,
ACB,1,1,1,1,4,4,1.6
MBB,2,2,2,2,2,0,2.0
VCB,4,3,3,6,5,0,4.2
BID,7,5,4,4,3,0,4.6
CTG,6,7,6,5,1,1,5.0
TCB,3,4,7,7,6,0,5.4
STB,8,8,5,3,8,0,6.4
VPB,5,6,8,8,7,0,6.8


■ Thứ hạng roa_nam


nam,2021,2022,2023,2024,2025,so_nam_hang_1,hang_tb
ma_nh,,,,,,,
TCB,1,1,3,1,1,4,1.4
MBB,3,3,1,2,3,1,2.4
VPB,2,2,6,4,2,0,3.2
ACB,4,4,2,3,4,0,3.4
VCB,5,5,4,5,5,0,4.8
CTG,6,6,7,7,6,0,6.4
STB,7,8,5,6,8,0,6.8
BID,8,7,8,8,7,0,7.6


In [ ]:
mt = pd.read_csv(KQ / "thong_ke_mo_ta.csv", encoding="utf-8-sig")
(mt[(mt["cap_do"] == "ma_nh") & mt["chi_so"].isin(["roa", "roe"])]
 .pivot(index="ma_nh", columns="chi_so", values=["trung_binh", "do_lech_chuan"]).round(2))

trung_binh        do_lech_chuan      
chi_so        roa    roe           roa   roe
ma_nh                                       
ACB          2.19  23.07          0.41  4.37
BID          0.93  17.88          0.22  4.16
CTG          1.12  17.82          0.35  5.43
MBB          2.46  23.26          0.38  3.48
STB          1.01  14.73          0.64  8.95
TCB          2.84  17.63          0.70  3.95
VCB          1.75  20.43          0.29  4.34
VPB          2.15  14.35          1.23  8.20

### 5.2 Câu 2 – Quốc doanh vs tư nhân

In [ ]:
kd = pd.read_csv(KQ / "kiem_dinh_nhom.csv", encoding="utf-8-sig")
kd[["chi_so", "phan_tich", "mau_du_lieu", "n_quoc_doanh", "n_tu_nhan", "tb_quoc_doanh", "tb_tu_nhan",
    "kiem_dinh", "p_value", "bac_bo_H0", "do_lon_hieu_ung"]].round(4)

,chi_so,phan_tich,mau_du_lieu,n_quoc_doanh,n_tu_nhan,tb_quoc_doanh,tb_tu_nhan,kiem_dinh,p_value,bac_bo_H0,do_lon_hieu_ung
0,nim,chinh,chinh,15,25,2.8486,4.6446,Welch t,0.0000,True,-1.6872
1,npl,chinh,chinh,15,25,1.2407,2.0569,Mann-Whitney U,0.2759,False,-0.2107
2,car,chinh,chinh,12,20,9.6542,12.6400,Mann-Whitney U,0.0003,True,-0.7833
3,npl,kiem_tra_lai,khong_VPB,15,20,1.2407,1.3921,Mann-Whitney U,0.9601,False,-0.0133
4,car,kiem_tra_lai,car_chi_so_vndirect,6,20,10.1533,12.6400,Student t,0.0240,True,-1.1212
5,nim,kiem_tra_lai,cap_ngan_hang,3,5,2.8486,4.6446,Mann-Whitney U,0.0357,True,-1.0000
6,npl,kiem_tra_lai,cap_ngan_hang,3,5,1.2407,2.0569,Mann-Whitney U,0.5714,False,-0.3333
7,car,kiem_tra_lai,cap_ngan_hang,3,5,9.6542,12.6400,Mann-Whitney U,0.0714,False,-0.8667


### 5.3 Câu 3 – Tương quan với lợi suất

In [ ]:
tq = pd.read_csv(KQ / "tuong_quan.csv", encoding="utf-8-sig")
(tq.pivot_table(index="chi_so", columns="mau_du_lieu", values="spearman_rho")
   .reindex(cf.DS_CHI_SO).round(3))

mau_du_lieu,cap_ngan_hang,car_cong_bo,chinh,loai_bien_dong_chung,q4_2021_2024,thay_doi_chi_so
chi_so,,,,,,
roa,0.214,NaN,0.030,0.082,-0.333,0.059
roe,-0.381,NaN,0.080,0.121,0.064,0.043
nim,0.595,NaN,-0.012,0.075,-0.477,-0.049
npl,0.595,NaN,0.036,0.004,-0.106,0.047
car,0.286,-0.515,-0.022,-0.039,-0.515,-0.021
ldr,0.238,NaN,-0.045,-0.006,-0.433,0.130


In [ ]:
tq[tq["phan_tich"] == "chinh"][["chi_so", "pearson_r", "pearson_p_hieu_chinh", "spearman_rho",
                                 "spearman_p_hieu_chinh", "xep_hang_do_manh"]].sort_values("xep_hang_do_manh").round(3)

,chi_so,pearson_r,pearson_p_hieu_chinh,spearman_rho,spearman_p_hieu_chinh,xep_hang_do_manh
1,roe,0.086,1.0,0.080,1.0,1
5,ldr,0.037,1.0,-0.045,1.0,2
3,npl,0.003,1.0,0.036,1.0,3
0,roa,0.090,1.0,0.030,1.0,4
4,car,0.007,1.0,-0.022,1.0,5
2,nim,0.085,1.0,-0.012,1.0,6


---
## 6. Danh sách bàn giao (Phần 7 – sản phẩm cuối)
| Sản phẩm | Vị trí |
|---|---|
| Notebook phân tích | `notebooks/01` → `06` |
| Hàm dùng chung | `src/config.py`, `data_loader.py`, `viz.py`, `thong_ke.py`, `kiem_tra.py`, `ket_qua.py` |
| 4 bảng kết quả (cho dashboard) | `outputs/thong_ke_mo_ta.csv`, `xep_hang_nam.csv`, `kiem_dinh_nhom.csv`, `tuong_quan.csv` |
| Bộ hình đầy đủ | `outputs/figures/p2_*` → `p5_*` |
| Bộ hình báo cáo (13 hình đã đánh số) | `outputs/figures/bao_cao/` |
| Mục báo cáo | "Kết quả trực quan hóa & phân tích thống kê" (doc riêng) |
| Quy ước cho dashboard | `QUY_UOC.md` |

In [ ]:
for f in ["thong_ke_mo_ta", "xep_hang_nam", "kiem_dinh_nhom", "tuong_quan"]:
    assert (KQ / f"{f}.csv").exists(), f
assert len(list((cf.THU_MUC_HINH / "bao_cao").glob("Hinh_*.png"))) == len(HINH_BAO_CAO)
print("Đủ sản phẩm bàn giao.")

Đủ sản phẩm bàn giao.
